# Day2課題: 社内文書RAGの検索設計

Word・Excel・PowerPoint・PDFを構造付きで抽出し、受講者が実装した**1つの検索パイプライン**を機械的に評価します。

検索器は`SearchEngine.search(query)`を実装し、最大上位5件を順位順に返します。共通評価コードは上位から合計2,500文字に制限し、40問の`required_facts`を本文から回収できた割合で**Recall@5**と**nDCG@5**を算出します。評価対象は1手法に限定し、回答評価APIは使いません。


## 1. 実行設定

リポジトリを置いた場所を `REPO_ROOT` に手入力し、deviceだけを初期設定します。


In [ ]:
from pathlib import Path

# 自分の環境に合わせて、estyle2026 リポジトリの場所を入力してください。
DATA_DIR = Path("data/enterprise_ai_policy_case")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"DATA_DIRが見つかりません: {DATA_DIR}\n"
        "REPO_ROOTを自分のestyle2026リポジトリの場所に変更してください。"
    )

print("DATA_DIR :", DATA_DIR)

import torch
from IPython.display import display
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("device:", DEVICE)

## 2. 実装

この課題で編集するのは、次の「受講者が編集するセル」の**2関数だけ**です。

- `extract_document(source_path)`：1ファイルを任意形式の抽出アイテムへ変換
- `build_search_engine(extracted_items)`：最大5件を返す`search(query)`を持つ検索器を構築

抽出ライブラリ・内部データ形式・検索方法は自由です。検索器の `search` は、根拠本文の文字列を順位順で返してください。文書名・ページ・スライド・シート名・idは返却契約に含めません。


In [ ]:
from typing import Any, Protocol


class SearchEngine(Protocol):
    """質問を受け取り、評価用の検索結果を順位順で返す検索器の共通契約。"""

    def search(self, query: str) -> list[str]:
        """質問queryに対する検索結果を文字列として最大5件返す。"""



### 受講者が編集するセル（この1セルだけ変更）

下の2関数以外は、原則として変更しません。


In [ ]:
# 参考実装（Baseline）: 編集しない
# MarkItDownで各ファイルを500文字の固定長chunkへ分割し、Chromaでベクトル検索します。
import unicodedata
from day2_exercise_evaluation import TOP_K

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from markitdown import MarkItDown


def normalize_text(text: object) -> str:
    """Baselineで用いる基本的な文字列正規化。"""
    return unicodedata.normalize("NFKC", str(text or "")).replace("\u00a0", " ").strip()


def extract_document(source_path: Path) -> list[Any]:
    """MarkItDownで入力ファイルをテキストへ変換し、500文字の固定長chunkへ分割する。"""
    text = normalize_text(MarkItDown().convert(source_path).text_content)
    return [
        {
            "record_id": f"{source_path.name}:{chunk_index}",
            "source_file": source_path.name,
            "text": text[start : start + 500],
        }
        for chunk_index, start in enumerate(range(0, len(text), 500), start=1)
    ]


def build_search_engine(extracted_items: list[Any]) -> SearchEngine:
    """ChromaとRuriの埋め込みを使い、最大5件を返すベクトル検索器を作る。"""
    records = {str(item["record_id"]): dict(item) for item in extracted_items}
    documents = [
        Document(page_content=normalize_text(item.get("text", "")), metadata={"record_id": record_id})
        for record_id, item in records.items()
    ]
    vector_store = Chroma(
        collection_name="day2_exercise_baseline_vector",
        embedding_function=HuggingFaceEmbeddings(
            model_name="cl-nagoya/ruri-v3-310m",
            model_kwargs={"device": DEVICE},
            encode_kwargs={"normalize_embeddings": True},
        ),
    )
    vector_store.add_documents(documents, ids=list(records))

    class ChromaVectorSearch:
        def search(self, query: str) -> list[str]:
            hits = vector_store.similarity_search(normalize_text(query), k=min(TOP_K, len(records)))
            return [
                str(records[hit.metadata["record_id"]].get("text", ""))
                for hit in hits
            ]

    return ChromaVectorSearch()




## 3. 実行と評価

実装後は、下の実行セルと評価セルを上から順に実行します。評価は、提出した**1つの手法**のRecall@5とnDCG@5です。

In [ ]:
from day2_exercise_evaluation import (
    evaluate_retrieval,
    load_questions,
    run_extraction,
    validate_search_engine,
)

# 配布コード: 編集しない
# 上の実装を定義した後に、このセルを実行します。
EXTRACTED_ITEMS = run_extraction(DATA_DIR, extract_document)
SEARCH_ENGINE = build_search_engine(EXTRACTED_ITEMS)
PIPELINE_NAME = type(SEARCH_ENGINE).__name__
validate_search_engine(SEARCH_ENGINE)

print(f"抽出アイテム数: {len(EXTRACTED_ITEMS)}")
print(f"検索器: {PIPELINE_NAME}")

questions_df = load_questions(DATA_DIR)
display(
    questions_df[
        ["question_id", "evaluation_group", "question", "required_facts"]
    ].head(8)
)

EVALUATION = evaluate_retrieval(DATA_DIR, SEARCH_ENGINE, PIPELINE_NAME)
retrieval_results_df = EVALUATION.results
retrieval_summary_df = EVALUATION.summary
retrieval_by_group_df = EVALUATION.by_group

display(retrieval_summary_df.round(3))
display(retrieval_by_group_df.round(3))
